# Week 3: Hidden State Probing - Let the LLM Tell Us

## The Core Insight

**Problem with current approach:**
- We manually classify tokens as "code" or "language" using keyword lists + external embeddings (all-MiniLM-L6-v2)
- This external model wasn't trained on code - it doesn't understand `useState` or `pandas` the way CodeLlama does
- We're essentially re-inventing what the LLM already knows!

**Key insight:**
When CodeLlama processes the prompt "How do I use pandas to read a CSV?", its hidden state at the final position **already encodes**:
- "I'm about to generate Python code"
- "This involves the pandas library"
- "The user wants a code example"

**The solution:**
Instead of manually classifying tokens, we probe the model's hidden states to ask: "What do you think you should generate next?"

---

## Approaches We'll Test

| Approach | Description | Uses LLM's Knowledge? |
|----------|-------------|----------------------|
| CCE (current) | Entropy over manually classified token buckets | No (external classification) |
| **Hidden State Probe** | Linear classifier on final hidden state | **Yes** |
| **LLM Token Embeddings** | Cluster/classify using CodeLlama's own embeddings | **Yes** |
| **Probability Distribution Shape** | Analyze top-k predictions directly | **Yes** |

---

## 1. Setup and Imports

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers torch accelerate scipy scikit-learn pandas matplotlib seaborn

In [ ]:
# Cell 2: Imports
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import entropy as scipy_entropy
from scipy.stats import ttest_ind
import time
from tqdm.notebook import tqdm

np.random.seed(42)
torch.manual_seed(42)
print("Imports successful")

## 2. Load CodeLlama Model

We load CodeLlama with `output_hidden_states=True` so we can access the internal representations.

In [ ]:
# Cell 3: Load Model
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)
model.eval()

print(f"Model loaded on {model.device}")
print(f"Vocabulary size: {len(tokenizer):,}")
print(f"Hidden size: {model.config.hidden_size}")
print(f"Number of layers: {model.config.num_hidden_layers}")

## 3. Test Examples

Same examples as before:
- **missing_context**: Questions asking for code where the model needs external API/library knowledge
- **language_choice**: Questions where the model explains existing code (has all context it needs)

In [ ]:
# Cell 4: Test Examples
TEST_EXAMPLES = [
    # Missing context: Model needs external knowledge about APIs/libraries
    {'id': 'code_1', 'type': 'missing_context', 'label': 1,
     'prompt': 'How do I use the requests library to make an HTTP GET request in Python? Show me the code.'},
    {'id': 'code_2', 'type': 'missing_context', 'label': 1,
     'prompt': 'Write a React component that uses useState. Show the import and component.'},
    {'id': 'code_3', 'type': 'missing_context', 'label': 1,
     'prompt': 'How do I authenticate with Firebase in a TypeScript project? Show the auth code.'},
    {'id': 'code_4', 'type': 'missing_context', 'label': 1,
     'prompt': 'Write a function that uses pandas to read a CSV file and filter rows.'},
    {'id': 'code_5', 'type': 'missing_context', 'label': 1,
     'prompt': 'How do I create a FastAPI endpoint that handles POST requests with JSON body?'},
    
    # Language choice: Model has all the context it needs (the code is in the prompt)
    {'id': 'lang_1', 'type': 'language_choice', 'label': 0,
     'prompt': 'Explain what this function does: def add(a, b): return a + b'},
    {'id': 'lang_2', 'type': 'language_choice', 'label': 0,
     'prompt': 'Write a comment describing this code: for item in items: process(item)'},
    {'id': 'lang_3', 'type': 'language_choice', 'label': 0,
     'prompt': 'Summarize this code: class User: def __init__(self, name): self.name = name'},
    {'id': 'lang_4', 'type': 'language_choice', 'label': 0,
     'prompt': 'Add a docstring to: def multiply(x, y): return x * y'},
    {'id': 'lang_5', 'type': 'language_choice', 'label': 0,
     'prompt': 'Describe this function: def is_even(n): return n % 2 == 0'},
]

print(f"{len(TEST_EXAMPLES)} test examples loaded")
print(f"  - {sum(1 for e in TEST_EXAMPLES if e['label'] == 1)} missing_context (label=1)")
print(f"  - {sum(1 for e in TEST_EXAMPLES if e['label'] == 0)} language_choice (label=0)")

## 4. Extract Hidden States

### What are Hidden States?

When the model processes a prompt, each layer produces a hidden state vector:

```
Prompt: "How do I use pandas?"
         ↓
    [Layer 0] → hidden_state_0 (4096 dims)
         ↓
    [Layer 1] → hidden_state_1 (4096 dims)
         ↓
       ....
         ↓
    [Layer 31] → hidden_state_31 (4096 dims)  ← This encodes "what to generate next"
         ↓
    [Output layer] → logits (32000 dims) → probability distribution
```

The **final layer's hidden state at the last token position** is the richest representation of "what the model thinks it should generate next".

### Why This Works

The hidden state encodes:
- The semantic meaning of the entire prompt
- Whether the model is in "code mode" or "language mode"
- The model's uncertainty about what comes next

We can train a simple linear classifier on these hidden states to detect "missing context"!

In [ ]:
# Cell 5: Extract hidden states for all examples

def extract_hidden_states(prompt: str, layer: int = -1) -> np.ndarray:
    """
    Extract the hidden state at the last token position.
    
    Args:
        prompt: The input prompt
        layer: Which layer to extract from (-1 = last layer)
    
    Returns:
        Hidden state vector of shape (hidden_size,) = (4096,)
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model(
            **inputs,
            output_hidden_states=True,
            return_dict=True
        )
    
    # Get hidden state from specified layer at last token position
    # outputs.hidden_states is a tuple of (n_layers + 1) tensors
    # Each tensor has shape (batch_size, seq_len, hidden_size)
    hidden_state = outputs.hidden_states[layer][:, -1, :]
    
    return hidden_state.squeeze().cpu().numpy()

def extract_multi_layer_hidden_states(prompt: str, layers: List[int] = [-1, -2, -3, -4]) -> np.ndarray:
    """
    Extract and concatenate hidden states from multiple layers.
    
    Different layers capture different levels of abstraction:
    - Earlier layers: syntactic features
    - Later layers: semantic features
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model(
            **inputs,
            output_hidden_states=True,
            return_dict=True
        )
    
    hidden_states = []
    for layer in layers:
        h = outputs.hidden_states[layer][:, -1, :].squeeze().cpu().numpy()
        hidden_states.append(h)
    
    return np.concatenate(hidden_states)

print("Hidden state extraction functions defined")
print(f"Single layer output: {model.config.hidden_size} dimensions")
print(f"Multi-layer (4 layers) output: {model.config.hidden_size * 4} dimensions")

In [ ]:
# Cell 6: Extract hidden states for all examples

print("Extracting hidden states from all examples...")
print("="*60)

hidden_states_last_layer = []
hidden_states_multi_layer = []
labels = []

for example in tqdm(TEST_EXAMPLES, desc="Processing"):
    # Extract single layer (last)
    h_single = extract_hidden_states(example['prompt'], layer=-1)
    hidden_states_last_layer.append(h_single)
    
    # Extract multi-layer (last 4 layers)
    h_multi = extract_multi_layer_hidden_states(example['prompt'], layers=[-1, -2, -3, -4])
    hidden_states_multi_layer.append(h_multi)
    
    labels.append(example['label'])
    
    print(f"{example['id']}: label={example['label']} ({example['type']})")

# Convert to numpy arrays
X_single = np.array(hidden_states_last_layer)
X_multi = np.array(hidden_states_multi_layer)
y = np.array(labels)

print("\n" + "="*60)
print(f"Single-layer features shape: {X_single.shape}")
print(f"Multi-layer features shape: {X_multi.shape}")
print(f"Labels shape: {y.shape}")
print(f"Class balance: {sum(y)}/{len(y)} = {sum(y)/len(y):.0%} positive")

## 5. Visualize Hidden States

Let's use PCA to visualize whether the hidden states naturally separate the two classes.

If the model already "knows" the difference between missing_context and language_choice prompts, we should see some clustering!

In [ ]:
# Cell 7: Visualize hidden states with PCA

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_single)

# Reduce to 2D for visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Plot
plt.figure(figsize=(10, 8))

colors = ['blue' if label == 0 else 'red' for label in y]
markers = ['o' if label == 0 else 's' for label in y]

for i, example in enumerate(TEST_EXAMPLES):
    plt.scatter(X_pca[i, 0], X_pca[i, 1], 
                c=colors[i], marker=markers[i], s=150, alpha=0.7)
    plt.annotate(example['id'], (X_pca[i, 0], X_pca[i, 1]), 
                 fontsize=9, ha='center', va='bottom')

# Add legend
plt.scatter([], [], c='blue', marker='o', s=100, label='language_choice (has context)')
plt.scatter([], [], c='red', marker='s', s=100, label='missing_context (needs context)')
plt.legend(loc='best', fontsize=10)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
plt.title('Hidden States (Last Layer) - PCA Projection\n'
          'Do the model\'s internal representations separate the two classes?', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('hidden_state_pca.png', dpi=150)
plt.show()

print(f"\nTotal variance explained by 2 PCs: {sum(pca.explained_variance_ratio_):.1%}")

## 6. Train Linear Probe

### What is a Linear Probe?

A linear probe is a simple logistic regression classifier trained on top of frozen hidden states. It tests whether the information we want (missing_context vs language_choice) is **linearly separable** in the model's representation space.

If a linear probe works well, it means:
1. The LLM's hidden states already encode this distinction
2. We don't need complex classification - the model knows!
3. The information is easily extractable

### Cross-Validation

With only 10 examples, we use Leave-One-Out cross-validation:
- Train on 9 examples, test on 1
- Repeat for all 10 examples
- Report average accuracy

In [ ]:
# Cell 8: Train linear probe with Leave-One-Out CV

print("Training Linear Probe on Hidden States")
print("="*60)

# Standardize features
scaler_single = StandardScaler()
X_single_scaled = scaler_single.fit_transform(X_single)

scaler_multi = StandardScaler()
X_multi_scaled = scaler_multi.fit_transform(X_multi)

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Test different configurations
configs = [
    ('Single Layer (last)', X_single_scaled),
    ('Multi-Layer (last 4)', X_multi_scaled),
]

results = []

for name, X in configs:
    # Logistic regression with L2 regularization
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    
    # Leave-One-Out CV
    scores = cross_val_score(clf, X, y, cv=loo, scoring='accuracy')
    
    print(f"\n{name}:")
    print(f"  LOO Accuracy: {scores.mean():.1%} (+/- {scores.std():.1%})")
    print(f"  Correct: {int(scores.sum())}/{len(scores)}")
    
    # Collect per-example predictions for analysis
    predictions = []
    probabilities = []
    for train_idx, test_idx in loo.split(X):
        clf_temp = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
        clf_temp.fit(X[train_idx], y[train_idx])
        pred = clf_temp.predict(X[test_idx])[0]
        prob = clf_temp.predict_proba(X[test_idx])[0]
        predictions.append(pred)
        probabilities.append(prob[1])  # Probability of class 1 (missing_context)
    
    results.append({
        'name': name,
        'accuracy': scores.mean(),
        'predictions': predictions,
        'probabilities': probabilities
    })

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
for r in results:
    print(f"{r['name']}: {r['accuracy']:.0%} accuracy")

In [ ]:
# Cell 9: Detailed prediction analysis

print("Detailed Prediction Analysis (Single Layer)")
print("="*60)

best_result = results[0]  # Single layer results

for i, example in enumerate(TEST_EXAMPLES):
    pred = best_result['predictions'][i]
    prob = best_result['probabilities'][i]
    true = example['label']
    correct = "correct" if pred == true else "WRONG"
    
    print(f"{example['id']:8s} | True: {true} | Pred: {pred} | P(missing)={prob:.3f} | {correct}")

print("\n" + "="*60)
print("Key Insight:")
print("  P(missing_context) > 0.5 → Probe thinks model needs external context")
print("  P(missing_context) < 0.5 → Probe thinks model has enough context")

## 7. Compare with CCE Approach

Now let's run the traditional CCE approach on the same examples and compare the results.

In [ ]:
# Cell 10: CCE computation (from original approach)

def softmax(logits: np.ndarray) -> np.ndarray:
    logits_stable = logits - np.max(logits)
    exp_logits = np.exp(logits_stable)
    return exp_logits / np.sum(exp_logits)

def shannon_entropy(probs: np.ndarray) -> float:
    probs = probs[probs > 0]  # Avoid log(0)
    return float(-np.sum(probs * np.log2(probs)))

# Simple keyword sets (from Week 1 - the approach that worked!)
CODE_KEYWORDS = {
    'if', 'else', 'for', 'while', 'def', 'class', 'return', 'import', 'from',
    'function', 'const', 'let', 'var', 'async', 'await', 'try', 'except',
    'print', 'console', 'log',
}

LANGUAGE_WORDS = {
    'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'been',
    'have', 'has', 'had', 'do', 'does', 'did',
    'what', 'how', 'why', 'when', 'where', 'which',
    'explain', 'describe', 'summarize', 'show', 'tell',
}

CODE_OPERATORS = {'+', '-', '*', '/', '=', '==', '!=', '<', '>', '{', '}', '[', ']', '(', ')', ';', ':'}

def classify_token(token: str) -> str:
    token_clean = token.strip().lower()
    if token.strip() in CODE_OPERATORS:
        return 'code'
    if token_clean in CODE_KEYWORDS:
        return 'code'
    if token_clean in LANGUAGE_WORDS:
        return 'language'
    return 'other'

# Pre-classify vocabulary
print("Pre-classifying vocabulary...")
vocab_classifications = {}
for token_id in tqdm(range(len(tokenizer)), desc="Classifying"):
    token_str = tokenizer.decode([token_id])
    vocab_classifications[token_id] = classify_token(token_str)

code_count = sum(1 for c in vocab_classifications.values() if c == 'code')
lang_count = sum(1 for c in vocab_classifications.values() if c == 'language')
print(f"\nVocabulary classification:")
print(f"  Code: {code_count:,} ({code_count/len(tokenizer):.1%})")
print(f"  Language: {lang_count:,} ({lang_count/len(tokenizer):.1%})")

In [ ]:
# Cell 11: Compute CCE for all examples

def compute_cce(prompt: str) -> Dict:
    """Compute CCE metrics for a prompt."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get logits for next token prediction
    logits = outputs.logits[:, -1, :].squeeze().cpu().numpy()
    probs = softmax(logits)
    
    # Separate by classification
    code_indices = [i for i, c in vocab_classifications.items() if c == 'code']
    lang_indices = [i for i, c in vocab_classifications.items() if c == 'language']
    
    code_probs = probs[code_indices]
    lang_probs = probs[lang_indices]
    
    # Normalize for entropy calculation
    code_probs_norm = code_probs / code_probs.sum() if code_probs.sum() > 0 else code_probs
    lang_probs_norm = lang_probs / lang_probs.sum() if lang_probs.sum() > 0 else lang_probs
    
    h_code = shannon_entropy(code_probs_norm)
    h_lang = shannon_entropy(lang_probs_norm)
    cce = h_code - h_lang
    
    return {
        'cce': cce,
        'h_code': h_code,
        'h_lang': h_lang,
        'code_prob_mass': float(code_probs.sum()),
        'lang_prob_mass': float(lang_probs.sum()),
    }

print("Computing CCE for all examples...")
print("="*60)

cce_results = []
for example in tqdm(TEST_EXAMPLES, desc="Computing CCE"):
    cce_data = compute_cce(example['prompt'])
    cce_results.append({
        'id': example['id'],
        'type': example['type'],
        'label': example['label'],
        **cce_data
    })
    print(f"{example['id']}: CCE={cce_data['cce']:+.3f}")

cce_df = pd.DataFrame(cce_results)
print("\nCCE computation complete")

## 8. Statistical Comparison

Compare:
1. **Hidden State Probe**: Classification accuracy
2. **CCE Approach**: Separation between groups (t-test, Cohen's d)

In [ ]:
# Cell 12: Statistical comparison

print("STATISTICAL COMPARISON")
print("="*70)

# Hidden State Probe Results
print("\n1. HIDDEN STATE PROBE (Single Layer)")
print("-"*50)
probe_accuracy = results[0]['accuracy']
probe_probs = results[0]['probabilities']

# Separate probabilities by true class
probe_probs_missing = [p for i, p in enumerate(probe_probs) if y[i] == 1]
probe_probs_language = [p for i, p in enumerate(probe_probs) if y[i] == 0]

print(f"LOO Accuracy: {probe_accuracy:.0%}")
print(f"Mean P(missing_context) for missing_context examples: {np.mean(probe_probs_missing):.3f}")
print(f"Mean P(missing_context) for language_choice examples: {np.mean(probe_probs_language):.3f}")

# t-test on probe probabilities
t_probe, p_probe = ttest_ind(probe_probs_missing, probe_probs_language)
print(f"t-statistic: {t_probe:.3f}")
print(f"p-value: {p_probe:.6f}")

# CCE Results
print("\n2. CCE APPROACH (Keyword Classification)")
print("-"*50)

cce_missing = cce_df[cce_df['label'] == 1]['cce'].values
cce_language = cce_df[cce_df['label'] == 0]['cce'].values

print(f"Mean CCE for missing_context: {cce_missing.mean():+.3f}")
print(f"Mean CCE for language_choice: {cce_language.mean():+.3f}")
print(f"Separation: {cce_missing.mean() - cce_language.mean():+.3f}")

# t-test on CCE
t_cce, p_cce = ttest_ind(cce_missing, cce_language)
print(f"t-statistic: {t_cce:.3f}")
print(f"p-value: {p_cce:.6f}")

# Cohen's d for CCE
pooled_std = np.sqrt(((len(cce_missing)-1)*cce_missing.std()**2 + 
                      (len(cce_language)-1)*cce_language.std()**2) / 
                     (len(cce_missing) + len(cce_language) - 2))
cohens_d = (cce_missing.mean() - cce_language.mean()) / pooled_std if pooled_std > 0 else 0
print(f"Cohen's d: {cohens_d:.3f}")

# CCE as classifier (threshold at 0)
cce_preds = [1 if cce > 0 else 0 for cce in cce_df['cce']]
cce_accuracy = sum(1 for i, p in enumerate(cce_preds) if p == cce_df.iloc[i]['label']) / len(cce_df)
print(f"Classification accuracy (threshold=0): {cce_accuracy:.0%}")

In [ ]:
# Cell 13: Visualization comparison

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Hidden State Probe Probabilities
ax1 = axes[0]
colors = ['red' if y[i] == 1 else 'blue' for i in range(len(y))]
bars = ax1.bar(range(len(probe_probs)), probe_probs, color=colors, alpha=0.7)
ax1.axhline(y=0.5, color='black', linestyle='--', linewidth=2, label='Decision threshold')
ax1.set_xlabel('Example', fontsize=12)
ax1.set_ylabel('P(missing_context)', fontsize=12)
ax1.set_title('Hidden State Probe\nProbability of "missing context"', fontsize=12)
ax1.set_xticks(range(len(TEST_EXAMPLES)))
ax1.set_xticklabels([e['id'] for e in TEST_EXAMPLES], rotation=45, ha='right')
ax1.legend()

# Add accuracy annotation
ax1.text(0.02, 0.98, f'Accuracy: {probe_accuracy:.0%}', 
         transform=ax1.transAxes, fontsize=11, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Plot 2: CCE Values
ax2 = axes[1]
cce_values = cce_df['cce'].values
bars = ax2.bar(range(len(cce_values)), cce_values, color=colors, alpha=0.7)
ax2.axhline(y=0, color='black', linestyle='--', linewidth=2, label='Decision threshold')
ax2.set_xlabel('Example', fontsize=12)
ax2.set_ylabel('CCE (code_entropy - lang_entropy)', fontsize=12)
ax2.set_title('CCE Approach\nContrastive Code Entropy', fontsize=12)
ax2.set_xticks(range(len(TEST_EXAMPLES)))
ax2.set_xticklabels([e['id'] for e in TEST_EXAMPLES], rotation=45, ha='right')
ax2.legend()

# Add accuracy annotation
ax2.text(0.02, 0.98, f'Accuracy: {cce_accuracy:.0%}', 
         transform=ax2.transAxes, fontsize=11, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Add legend for colors
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='red', alpha=0.7, label='missing_context (true)'),
                   Patch(facecolor='blue', alpha=0.7, label='language_choice (true)')]
fig.legend(handles=legend_elements, loc='upper center', ncol=2, fontsize=10)

plt.tight_layout()
plt.subplots_adjust(top=0.88)
plt.savefig('probe_vs_cce_comparison.png', dpi=150)
plt.show()

## 9. Bonus: Analyze What the Probe Learned

Let's look at the probe's weights to understand what directions in hidden state space correspond to "missing context".

In [ ]:
# Cell 14: Analyze probe weights

# Train final probe on all data
final_probe = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
final_probe.fit(X_single_scaled, y)

# Get weights
weights = final_probe.coef_[0]

print("Probe Weight Analysis")
print("="*60)
print(f"Weight vector shape: {weights.shape}")
print(f"Weight magnitude (L2 norm): {np.linalg.norm(weights):.4f}")
print(f"Max weight: {weights.max():.4f}")
print(f"Min weight: {weights.min():.4f}")
print(f"\nInterpretation:")
print(f"  Large positive weights → hidden state dimensions that indicate 'missing context'")
print(f"  Large negative weights → hidden state dimensions that indicate 'has context'")

# Top positive and negative weights
top_positive = np.argsort(weights)[-10:]
top_negative = np.argsort(weights)[:10]

print(f"\nTop 10 'missing context' dimensions: {top_positive}")
print(f"Top 10 'has context' dimensions: {top_negative}")

In [ ]:
# Cell 15: Weight distribution visualization

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(weights, bins=50, color='steelblue', alpha=0.7)
plt.axvline(x=0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Weight Value', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Distribution of Probe Weights', fontsize=12)

plt.subplot(1, 2, 2)
sorted_weights = np.sort(weights)[::-1]
plt.plot(sorted_weights, color='steelblue', linewidth=1.5)
plt.axhline(y=0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Dimension (sorted)', fontsize=12)
plt.ylabel('Weight Value', fontsize=12)
plt.title('Sorted Probe Weights', fontsize=12)

plt.tight_layout()
plt.savefig('probe_weights.png', dpi=150)
plt.show()

# Sparsity analysis
threshold = 0.01
n_important = np.sum(np.abs(weights) > threshold)
print(f"\nDimensions with |weight| > {threshold}: {n_important} ({n_important/len(weights):.1%} of hidden state)")
print(f"This suggests the 'missing context' signal is {'distributed' if n_important > 100 else 'concentrated'} across the hidden state.")

## 10. Summary and Conclusions

In [ ]:
# Cell 16: Final summary

print("="*70)
print("SUMMARY: Hidden State Probe vs CCE")
print("="*70)

print("\n| Approach               | Accuracy | p-value  | Key Insight                    |")
print("|------------------------|----------|----------|--------------------------------|")
print(f"| Hidden State Probe     | {probe_accuracy:>6.0%}   | {p_probe:<8.4f} | Uses LLM's own representations |")
print(f"| CCE (keyword classify) | {cce_accuracy:>6.0%}   | {p_cce:<8.4f} | Uses external token classification |")

print("\n" + "-"*70)
print("KEY INSIGHTS:")
print("-"*70)

if probe_accuracy >= cce_accuracy:
    print("\n1. The Hidden State Probe performs at least as well as CCE.")
    print("   This confirms: The LLM ALREADY encodes 'missing context' information!")
else:
    print("\n1. CCE performs better in this test.")
    print("   However, probe approach may improve with more training data.")

print("\n2. ADVANTAGES of Hidden State Probe:")
print("   - No manual token classification needed")
print("   - Uses the LLM's own understanding of context")
print("   - Single forward pass (no vocabulary scanning)")
print("   - More interpretable (probe weights show what dimensions matter)")

print("\n3. ADVANTAGES of CCE:")
print("   - No training required (threshold-based)")
print("   - Interpretable metric (entropy over code vs language)")
print("   - Works with any tokenizer vocabulary")

print("\n4. RECOMMENDATION:")
if probe_accuracy >= 0.8:
    print("   The hidden state approach is promising! Consider:")
    print("   - Collecting more training examples")
    print("   - Testing on held-out data")
    print("   - Comparing across different prompt types")
else:
    print("   Results are inconclusive with only 10 examples.")
    print("   Need more data to properly evaluate the probe approach.")

print("\n" + "="*70)

In [ ]:
# Cell 17: Save results

# Combine results
final_results = []
for i, example in enumerate(TEST_EXAMPLES):
    final_results.append({
        'id': example['id'],
        'type': example['type'],
        'true_label': example['label'],
        'prompt': example['prompt'],
        # Probe results
        'probe_prob': results[0]['probabilities'][i],
        'probe_pred': results[0]['predictions'][i],
        'probe_correct': results[0]['predictions'][i] == example['label'],
        # CCE results
        'cce': cce_df.iloc[i]['cce'],
        'cce_pred': 1 if cce_df.iloc[i]['cce'] > 0 else 0,
        'cce_correct': (1 if cce_df.iloc[i]['cce'] > 0 else 0) == example['label'],
    })

results_df = pd.DataFrame(final_results)
results_df.to_csv('week3_hidden_state_probe_results.csv', index=False)

print("Results saved to week3_hidden_state_probe_results.csv")
print("\nFinal Results:")
print(results_df[['id', 'true_label', 'probe_prob', 'probe_correct', 'cce', 'cce_correct']].to_string())

---

## Next Steps

Based on these results, consider:

1. **If probe works well:**
   - Collect more training examples (50-100) for robust evaluation
   - Test on different model sizes (CodeLlama-13B, CodeLlama-34B)
   - Explore which layers are most informative

2. **Combine approaches:**
   - Use probe probability + CCE as ensemble
   - Probe for coarse detection, CCE for fine-grained analysis

3. **Production considerations:**
   - Probe: Single forward pass, ~4KB weights to store
   - CCE: No training needed, but requires vocabulary classification

---